# D3QN+PER Quantum Circuit Routing — Training Notebook

This notebook trains the D3QN+PER agent on Google Colab with A100 GPU.
All outputs (checkpoints, logs, figures, evals) are saved to Google Drive.

**Features:**
- Auto-resumes from interrupted training runs
- Saves everything to Google Drive (survives disconnects)
- Rich console output with metrics every 25 episodes
- Periodic evaluation against SABRE baseline

**Training phases** (run one at a time):
1. `linear5` — Sanity check (~5 min)
2. `heavy_hex` — Primary experiment (~45-60 min on A100)
3. `multi` — Cross-topology generalization (~1.5-2 hrs on A100)

---
## 1. Setup: Mount Drive, Clone Repo, Install Dependencies

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

# === CONFIGURE THESE ===
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/rl-quantum-routing"  # Where to save all outputs
REPO_URL = "https://github.com/helloelora/rl-quantum-circuit-routing.git"
BRANCH = "ali-dev"
# =======================

# Create project directory on Drive
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"Drive project dir: {DRIVE_PROJECT_DIR}")
print(f"Contents: {os.listdir(DRIVE_PROJECT_DIR)}")

Drive project dir: /content/drive/MyDrive/rl-quantum-routing
Contents: []


In [6]:
# Clone or update repo
REPO_DIR = "/content/rl-quantum-circuit-routing"

if os.path.exists(REPO_DIR):
    print("Repo exists, pulling latest...")
    !cd {REPO_DIR} && git pull origin {BRANCH}
else:
    print("Cloning repo...")
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

print(f"\nLatest commit:")
!cd {REPO_DIR} && git log --oneline -3

Repo exists, pulling latest...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.54 KiB | 1.54 MiB/s, done.
From https://github.com/helloelora/rl-quantum-circuit-routing
 * branch            ali-dev    -> FETCH_HEAD
   9ec95d6..50ba574  ali-dev    -> origin/ali-dev
Updating 9ec95d6..50ba574
Fast-forward
 notebooks/train_colab.ipynb | 112 +++++++++++++++++++++++++++++++++++---------
 1 file changed, 89 insertions(+), 23 deletions(-)

Latest commit:
50ba574 (HEAD -> ali-dev, origin/ali-dev) Fix Colab notebook: handle PyTorch 2.10 API change and relax deps
9ec95d6 Update presets with recommended training params
34a32d7 Fix resume to reuse existing run directory instead of creating new one


In [ ]:
# Install dependencies
# IMPORTANT: Don't pin numpy — Colab has numpy 2.x and other packages depend on it.
# Use qiskit>=1.0 (not ==1.0.2) to avoid forcing a numpy downgrade.
!pip install -q "qiskit>=1.0" gymnasium==0.29.1 networkx matplotlib imageio>=2.31.0

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"GPU Memory: {mem / 1e9:.1f} GB")

import numpy as np
print(f"NumPy: {np.__version__}")

In [8]:
# Add src/ to Python path
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

# Quick import test
from config import TrainConfig, linear5_sanity_config, heavy_hex_config, multi_topology_config
from environment import QubitRoutingEnv
from dqn_agent import D3QNAgent
print("All imports OK")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

---
## 2. Choose Training Phase

Run **one** of the cells below to set the config. Each phase is independent (fresh start).

In [ ]:
# ============================
# PHASE 1: Sanity check (linear_5)
# ~5 minutes, verifies the pipeline works
# ============================
PHASE = "linear5"
config = linear5_sanity_config()
config.log_every = 25          # Print more often
config.eval_episodes = 10
config.device = "cuda"         # Use GPU

print(f"Phase: {PHASE}")
print(f"Topology: {config.topologies}")
print(f"Episodes: {config.total_episodes}")
print(f"Circuit depth: {config.circuit_depth}")

In [ ]:
# ============================
# PHASE 2: Primary training (heavy_hex_19)
# ~45-60 min on A100
# Preset includes: circuit_depth=30, eval_episodes=50, log_every=25
# ============================
PHASE = "heavy_hex"
config = heavy_hex_config()
config.device = "cuda"

print(f"Phase: {PHASE}")
print(f"Topology: {config.topologies}")
print(f"Episodes: {config.total_episodes}")
print(f"Circuit depth: {config.circuit_depth}")
print(f"Eval episodes: {config.eval_episodes}")
print(f"Log every: {config.log_every}")

In [ ]:
# ============================
# PHASE 3: Multi-topology (novel contribution)
# ~1.5-2 hours on A100
# Preset includes: circuit_depth=30, eval_episodes=50, log_every=25
# ============================
PHASE = "multi"
config = multi_topology_config()
config.device = "cuda"

print(f"Phase: {PHASE}")
print(f"Topologies: {config.topologies}")
print(f"Episodes: {config.total_episodes}")
print(f"Circuit depth: {config.circuit_depth}")
print(f"Eval episodes: {config.eval_episodes}")
print(f"Log every: {config.log_every}")

---
## 3. Auto-Resume Logic

Checks Google Drive for an existing run of the same phase. If found, resumes from the latest checkpoint. Otherwise starts a new run.

**How it works:**
1. Looks in `DRIVE_PROJECT_DIR/{PHASE}/` for existing run directories
2. In the latest run dir, finds the most recent checkpoint
3. If found → loads that checkpoint and continues training from that episode
4. If not found → starts a fresh run with a new run number

In [ ]:
from pathlib import Path
import glob
import re

# Output base on Drive, per phase
phase_output_dir = os.path.join(DRIVE_PROJECT_DIR, PHASE)
config.output_base = phase_output_dir

# --- Find existing runs for this phase ---
resume_checkpoint = None

if os.path.exists(phase_output_dir):
    run_dirs = sorted(glob.glob(os.path.join(phase_output_dir, "run_*")))
    if run_dirs:
        latest_run = run_dirs[-1]
        ckpt_dir = os.path.join(latest_run, "checkpoints")

        if os.path.exists(ckpt_dir):
            # Find all checkpoint files, prefer non-emergency ones
            ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "checkpoint_ep*.pt")))
            final_ckpt = os.path.join(ckpt_dir, "checkpoint_final.pt")
            emergency_ckpt = os.path.join(ckpt_dir, "checkpoint_emergency.pt")

            if os.path.exists(final_ckpt):
                # Training completed — don't resume, will start new run
                print(f"Previous run completed: {latest_run}")
                print(f"Starting a NEW run (previous one finished).")
            elif ckpts:
                # Has periodic checkpoints — resume from latest
                resume_checkpoint = ckpts[-1]
                # Extract episode number from filename
                match = re.search(r"checkpoint_ep(\d+)\.pt", resume_checkpoint)
                resume_ep = int(match.group(1)) if match else "?"
                print(f"RESUMING from: {resume_checkpoint}")
                print(f"  Run dir: {latest_run}")
                print(f"  Episode: {resume_ep}")
            elif os.path.exists(emergency_ckpt):
                # Only emergency checkpoint exists
                resume_checkpoint = emergency_ckpt
                print(f"RESUMING from emergency checkpoint: {resume_checkpoint}")
                print(f"  Run dir: {latest_run}")
            else:
                print(f"Run dir exists but no checkpoints: {latest_run}")
                print("Starting fresh.")
        else:
            print(f"Run dir exists but no checkpoint dir: {latest_run}")
            print("Starting fresh.")
    else:
        print(f"No previous runs found in {phase_output_dir}")
        print("Starting fresh.")
else:
    print(f"Phase directory doesn't exist yet: {phase_output_dir}")
    print("Starting fresh.")

if resume_checkpoint:
    # Load config from the existing run to stay consistent
    existing_config_path = os.path.join(os.path.dirname(os.path.dirname(resume_checkpoint)), "config.json")
    if os.path.exists(existing_config_path):
        print(f"\nLoading config from existing run: {existing_config_path}")
        config = TrainConfig.load(existing_config_path)
        config.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"  Episodes: {config.total_episodes}")
        print(f"  Topologies: {config.topologies}")

---
## 4. Train

This cell runs the full training loop. Everything is saved to Google Drive as it runs:
- **Logs** (episodes.jsonl, train_steps.jsonl, evaluations.jsonl) — appended continuously
- **Checkpoints** — saved every `checkpoint_every` episodes
- **Figures** — training curves updated at each eval, eval comparison charts per checkpoint
- **Eval results** — full JSON with per-circuit results + trajectories

If the runtime disconnects, re-run from the top — it will auto-resume from the latest checkpoint.

In [ ]:
import time
import numpy as np

# Print full config
print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
print(f"  Phase:           {PHASE}")
print(f"  Topologies:      {config.topologies}")
print(f"  Circuit depth:   {config.circuit_depth}")
print(f"  Max steps/ep:    {config.max_steps}")
print(f"  Total episodes:  {config.total_episodes}")
print(f"  Batch size:      {config.batch_size}")
print(f"  Learning rate:   {config.lr}")
print(f"  Buffer capacity: {config.buffer_capacity}")
print(f"  Gamma:           {config.gamma}")
print(f"  Epsilon:         {config.epsilon_start} → {config.epsilon_end} over {config.epsilon_decay_steps} steps")
print(f"  PER alpha:       {config.per_alpha}")
print(f"  PER beta:        {config.per_beta_start} → {config.per_beta_end} over {config.per_beta_anneal_steps} steps")
print(f"  Target update:   every {config.target_update_freq} grad steps")
print(f"  Eval every:      {config.eval_every} episodes ({config.eval_episodes} circuits/eval)")
print(f"  Checkpoint every: {config.checkpoint_every} episodes")
print(f"  Log every:       {config.log_every} episodes")
print(f"  Device:          {config.device}")
print(f"  Seed:            {config.seed}")
print(f"  Output base:     {config.output_base}")
print(f"  Resume from:     {resume_checkpoint or 'None (fresh start)'}")
print("=" * 60)

# Confirm GPU
if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
else:
    print("\nWARNING: No GPU detected! Training will be slow.")

print(f"\nStarting training...\n")

In [ ]:
from train import train

t0 = time.time()
train(config, resume_from=resume_checkpoint)
elapsed = time.time() - t0

print(f"\n{'=' * 60}")
print(f"TRAINING COMPLETE")
print(f"Total time: {elapsed/60:.1f} minutes ({elapsed/3600:.2f} hours)")
print(f"Output dir: {config.run_dir}")
print(f"{'=' * 60}")

---
## 5. Post-Training: View Results

Display the training curves and evaluation results right here in the notebook.

In [ ]:
from IPython.display import display, Image as IPImage

# Show training curves
curves_path = os.path.join(config.figures_dir, "training_curves.png")
if os.path.exists(curves_path):
    print("Training Curves:")
    display(IPImage(filename=curves_path))
else:
    print(f"No training curves found at {curves_path}")

In [ ]:
# Show latest eval comparison
import glob as _glob

eval_figs = sorted(_glob.glob(os.path.join(config.figures_dir, "eval_comparison_ep*.png")))
if eval_figs:
    latest_fig = eval_figs[-1]
    print(f"Latest eval comparison ({os.path.basename(latest_fig)}):")
    display(IPImage(filename=latest_fig))
else:
    print("No eval comparison figures found.")

In [ ]:
# Print final eval summary
import json

eval_jsons = sorted(_glob.glob(os.path.join(config.eval_dir, "eval_ep*.json")))
if eval_jsons:
    latest_eval = eval_jsons[-1]
    with open(latest_eval) as f:
        data = json.load(f)
    summary = data.get("summary", {})
    print(f"Latest evaluation ({os.path.basename(latest_eval)}):")
    print(f"  Completion rate:    {summary.get('completion_rate', 0):.0%}")
    print(f"  Mean agent SWAPs:   {summary.get('mean_agent_swaps', 0):.1f}")
    print(f"  Mean SABRE SWAPs:   {summary.get('mean_sabre_swaps', 0):.1f}")
    print(f"  Mean swap ratio:    {summary.get('mean_swap_ratio', 0):.3f}")
    print(f"  Median swap ratio:  {summary.get('median_swap_ratio', 0):.3f}")
    print(f"  Mean reward:        {summary.get('mean_reward', 0):.2f}")
else:
    print("No eval results found.")

In [ ]:
# Show eval progression across training
eval_log_path = os.path.join(config.log_dir, "evaluations.jsonl")
if os.path.exists(eval_log_path):
    evals = []
    with open(eval_log_path) as f:
        for line in f:
            line = line.strip()
            if line:
                evals.append(json.loads(line))

    if evals:
        print(f"{'Episode':>8} | {'Completion':>10} | {'Agent SWAPs':>12} | {'SABRE SWAPs':>12} | {'Ratio':>7}")
        print("-" * 65)
        for e in evals:
            print(
                f"{e.get('episode', '?'):>8} | "
                f"{e.get('completion_rate', 0):>9.0%} | "
                f"{e.get('mean_agent_swaps', 0):>12.1f} | "
                f"{e.get('mean_sabre_swaps', 0):>12.1f} | "
                f"{e.get('mean_swap_ratio', 0):>7.3f}"
            )
    else:
        print("No eval records yet.")
else:
    print("No evaluations log found.")

---
## 6. Generate Visualization GIFs (Optional)

Generate step-by-step routing animations from the latest eval trajectories.

In [ ]:
from visualize import create_routing_gif, create_side_by_side_gif

# Find latest eval with trajectories
gif_dir = os.path.join(config.figures_dir, "gifs")
os.makedirs(gif_dir, exist_ok=True)

eval_jsons = sorted(_glob.glob(os.path.join(config.eval_dir, "eval_ep*.json")))
if eval_jsons:
    with open(eval_jsons[-1]) as f:
        data = json.load(f)

    trajectories = data.get("trajectories", [])
    if trajectories:
        n_gifs = min(5, len(trajectories))  # Up to 5 GIFs
        print(f"Generating {n_gifs} routing GIFs...")
        for i in range(n_gifs):
            traj = trajectories[i]
            create_routing_gif(
                traj, os.path.join(gif_dir, f"routing_{i}.gif"), fps=1
            )
            create_side_by_side_gif(
                traj, os.path.join(gif_dir, f"routing_vs_sabre_{i}.gif"), fps=1
            )
        print(f"\nGIFs saved to: {gif_dir}")
    else:
        print("No trajectories in eval data (need log_trajectories=True).")
else:
    print("No eval results found.")

---
## 7. List All Outputs on Drive

In [ ]:
# Show what's saved on Drive
print(f"Drive project directory: {DRIVE_PROJECT_DIR}\n")

for root, dirs, files in os.walk(DRIVE_PROJECT_DIR):
    level = root.replace(DRIVE_PROJECT_DIR, "").count(os.sep)
    indent = "  " * level
    folder = os.path.basename(root)
    print(f"{indent}{folder}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        size = os.path.getsize(os.path.join(root, file))
        if size > 1e6:
            size_str = f"{size/1e6:.1f} MB"
        elif size > 1e3:
            size_str = f"{size/1e3:.1f} KB"
        else:
            size_str = f"{size} B"
        print(f"{sub_indent}{file}  ({size_str})")

---
## 8. Download Checkpoint Locally (Optional)

If you want to copy the final checkpoint to your local machine for further evaluation.

In [ ]:
# Download the final checkpoint
from google.colab import files

final_ckpt = os.path.join(config.checkpoint_dir, "checkpoint_final.pt")
if os.path.exists(final_ckpt):
    print(f"Downloading: {final_ckpt}")
    files.download(final_ckpt)
else:
    # Try latest periodic checkpoint
    ckpts = sorted(_glob.glob(os.path.join(config.checkpoint_dir, "checkpoint_ep*.pt")))
    if ckpts:
        print(f"No final checkpoint. Downloading latest: {ckpts[-1]}")
        files.download(ckpts[-1])
    else:
        print("No checkpoints found.")